In [ ]:
# ============================================================
# FINAL PRODUCTION ML PIPELINE – MICRO‑INVESTING SEGMENTATION
# ============================================================
# Behavioural clusters only (savings & expense ratios).
# 3 product segments enforced.
# Safety guardrails protect against dangerous recommendations.
# ============================================================

import os, json, warnings, numpy as np, pandas as pd
import kagglehub
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
import joblib
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.switch_backend('Agg')                     # non-interactive backend

ARTIFACT_DIR = "/content/micro_investing_artifacts"
K_CLUSTERS   = 3                               # product decision

# ============================================================
# 1. HELPERS
# ============================================================
def compute_total_expenses(row: dict) -> float:
    """Sum all spending categories to get total monthly outflow."""
    cats = ["Rent","Loan_Repayment","Insurance","Groceries","Transport",
            "Eating_Out","Entertainment","Utilities","Healthcare","Education",
            "Miscellaneous"]
    return sum(row.get(c, 0.0) for c in cats)

def compute_savings(income, total_expenses):
    return income - total_expenses

def savings_ratio(income, total_expenses):
    return (income - total_expenses) / (income + 1e-9)

def expense_ratio(income, total_expenses):
    return total_expenses / (income + 1e-9)

# ============================================================
# 2. TRAINING PIPELINE
# ============================================================
def train_pipeline():
    os.makedirs(ARTIFACT_DIR, exist_ok=True)

    print("Downloading/loading dataset...")
    path = kagglehub.dataset_download("shriyashjagtap/indian-personal-finance-and-spending-habits")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df = pd.read_csv(os.path.join(path, csv_file))
    print(f"Dataset shape: {df.shape}")

    # === FEATURE ENGINEERING ===
    # Compute total expenses and savings from category columns if possible,
    # else fall back to provided columns.
    if "Rent" in df.columns:
        df['Total_Expenses'] = df.apply(lambda row: compute_total_expenses(row), axis=1)
    else:
        if 'Total_Expenses' not in df.columns:
            raise KeyError("Cannot compute Total_Expenses – missing columns")
    df['Income'] = df.get('Income', 0)
    df['Savings'] = df['Income'] - df['Total_Expenses']

    # Behavioural features only – NO raw income
    df['Savings_Ratio'] = savings_ratio(df['Income'], df['Total_Expenses'])
    df['Expense_Ratio'] = expense_ratio(df['Income'], df['Total_Expenses'])

    X = df[['Savings_Ratio','Expense_Ratio']].copy()
    # Clip impossible values (sanity)
    X['Savings_Ratio'] = X['Savings_Ratio'].clip(-1, 1.5)
    X['Expense_Ratio'] = X['Expense_Ratio'].clip(0, 1.5)

    # Impute (should not be needed, but safety)
    imputer = SimpleImputer(strategy='median')
    X_imp = imputer.fit_transform(X)

    # Scale
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_imp)

    # === CLUSTERING ===
    print(f"\nTraining KMeans with {K_CLUSTERS} behavioural clusters...")
    kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init=10)
    df['cluster_id'] = kmeans.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, df['cluster_id'])
    print(f"Behavioural Silhouette Score: {sil:.4f}")

    # Cluster profiles (for documentation)
    profile_df = df.groupby('cluster_id').agg(
        mean_income=('Income','mean'),
        median_income=('Income','median'),
        mean_expense=('Total_Expenses','mean'),
        mean_savings_ratio=('Savings_Ratio','mean'),
        median_savings_ratio=('Savings_Ratio','median'),
        mean_expense_ratio=('Expense_Ratio','mean')
    ).round(4)
    print("\nCluster profiles:")
    print(profile_df)

    # === DYNAMIC PERSONA MAPPING ===
    # Sort clusters by median Savings Ratio: lowest = Stressed,
    # middle = Cautious, highest = Investment Ready.
    median_sr = profile_df['median_savings_ratio']
    sorted_clusters = median_sr.sort_values().index.tolist()

    SEGMENT_MAP = {
        int(sorted_clusters[0]): {
            "name": "Financially Stressed",
            "plan": "Debt Management & Stabilization",
            "multiplier": 0.0,
            "investment_appetite": "Zero (Build emergency fund)"
        },
        int(sorted_clusters[1]): {
            "name": "Cautious Saver",
            "plan": "Conservative SIP (Index/Liquid Funds)",
            "multiplier": 0.2,
            "investment_appetite": "Low-to-Medium"
        },
        int(sorted_clusters[2]): {
            "name": "Investment Ready",
            "plan": "Moderate-to-Aggressive SIP",
            "multiplier": 0.4,
            "investment_appetite": "High"
        }
    }

    # === SAVE ARTIFACTS ===
    joblib.dump(imputer, os.path.join(ARTIFACT_DIR, 'imputer.pkl'))
    joblib.dump(scaler,  os.path.join(ARTIFACT_DIR, 'robust_scaler.pkl'))
    joblib.dump(kmeans,  os.path.join(ARTIFACT_DIR, 'kmeans_model.pkl'))
    with open(os.path.join(ARTIFACT_DIR, 'segment_map.json'), 'w') as f:
        json.dump(SEGMENT_MAP, f, indent=2)
    # save feature order for prediction
    with open(os.path.join(ARTIFACT_DIR, 'feature_order.json'), 'w') as f:
        json.dump(['Savings_Ratio','Expense_Ratio'], f)

    print(f"\n✅ Artifacts saved to: {ARTIFACT_DIR}")
    print("✅ ML PART COMPLETED")
    return df, SEGMENT_MAP, kmeans, scaler, imputer

# ============================================================
# 3. INFERENCE CLASS
# ============================================================
class MicroInvestmentAssistant:
    def __init__(self, artifacts_dir=ARTIFACT_DIR):
        self.imputer = joblib.load(os.path.join(artifacts_dir, 'imputer.pkl'))
        self.scaler  = joblib.load(os.path.join(artifacts_dir, 'robust_scaler.pkl'))
        self.kmeans  = joblib.load(os.path.join(artifacts_dir, 'kmeans_model.pkl'))
        with open(os.path.join(artifacts_dir, 'segment_map.json')) as f:
            self.segment_map = json.load(f)
        # Convert segment_map keys back to int (json saves as str)
        self.segment_map = {int(k): v for k,v in self.segment_map.items()}
        with open(os.path.join(artifacts_dir, 'feature_order.json')) as f:
            self.feature_order = json.load(f)

    def _preprocess(self, user_input: dict) -> np.ndarray:
        income = user_input.get('Income', 0)
        total_expenses = compute_total_expenses(user_input)
        sav   = income - total_expenses
        sr    = savings_ratio(income, total_expenses)
        er    = expense_ratio(income, total_expenses)
        row   = np.array([[sr, er]])
        row_imp = self.imputer.transform(row)
        row_sc  = self.scaler.transform(row_imp)
        return row_sc, income, total_expenses, sav

    def recommend(self, user_input: dict) -> dict:
        X_sc, income, total_expenses, savings = self._preprocess(user_input)
        cluster_id = int(self.kmeans.predict(X_sc)[0])

        # SAFETY GUARDRAILS
        reasons = []
        multiplier_override = None
        segment_override = None

        if savings <= 0 or total_expenses >= income:
            reasons.append("CRITICAL: Negative/Zero savings. Forcing Stressed.")
            # force to the stressed cluster (lowest savings ratio)
            stressed_cluster = min(self.segment_map.keys(),
                                   key=lambda c: self.segment_map[c]['multiplier'])
            segment_override = stressed_cluster
            multiplier_override = 0.0
        elif savings_ratio(income, total_expenses) < 0.10:
            reasons.append("WARNING: Savings ratio under 10%. Capping risk.")
            # force to cautious cluster (middle)
            cautious_cluster = sorted(self.segment_map.keys(),
                                      key=lambda c: self.segment_map[c]['multiplier'])[1]
            segment_override = cautious_cluster
            multiplier_override = 0.1   # very conservative

        cluster_used = segment_override if segment_override is not None else cluster_id
        seg_info = self.segment_map[cluster_used].copy()
        multiplier = multiplier_override if multiplier_override is not None else seg_info['multiplier']
        suggested_investment = round(savings * multiplier, 2)

        return {
            "cluster_id": cluster_used,
            "user_segment": seg_info['name'],
            "recommended_plan": seg_info['plan'],
            "monthly_income": float(income),
            "monthly_expenses": float(total_expenses),
            "monthly_savings": float(savings),
            "suggested_monthly_investment": suggested_investment,
            "investment_appetite": seg_info['investment_appetite'],
            "reason_codes": reasons
        }

    def confidence(self, user_input: dict) -> float:
        X_sc, _, _, _ = self._preprocess(user_input)
        dists = self.kmeans.transform(X_sc)[0]
        sorted_dists = np.sort(dists)
        if len(sorted_dists) < 2:
            return 0.5
        best = sorted_dists[0]
        second = sorted_dists[1]
        # Ratio of second to (best+second) -> higher when best is much smaller
        conf = second / (best + second + 1e-9)
        return float(np.clip(conf, 0.0, 1.0))

# ============================================================
# 4. SANITY TESTS
# ============================================================
def run_sanity_tests():
    assistant = MicroInvestmentAssistant()

    test_cases = [
        ("Low Income, High Debt", {
            "Income": 15000,
            "Rent": 5000, "Loan_Repayment": 6000, "Insurance": 500,
            "Groceries": 2000, "Transport": 500, "Eating_Out": 500,
            "Entertainment": 200, "Utilities": 300, "Healthcare": 200,
            "Education": 0, "Miscellaneous": 300, "Dependents": 2
        }),
        ("Mid Income, Balanced", {
            "Income": 45000,
            "Rent": 10000, "Loan_Repayment": 3000, "Insurance": 1000,
            "Groceries": 4000, "Transport": 2000, "Eating_Out": 2000,
            "Entertainment": 1000, "Utilities": 1500, "Healthcare": 500,
            "Education": 1000, "Miscellaneous": 500, "Dependents": 1
        }),
        ("High Income, Low Expense", {
            "Income": 120000,
            "Rent": 20000, "Loan_Repayment": 5000, "Insurance": 3000,
            "Groceries": 6000, "Transport": 3000, "Eating_Out": 3000,
            "Entertainment": 2000, "Utilities": 2000, "Healthcare": 1000,
            "Education": 2000, "Miscellaneous": 1000, "Dependents": 0
        }),
        ("High Income, Overspender", {
            "Income": 150000,
            "Rent": 40000, "Loan_Repayment": 30000, "Insurance": 5000,
            "Groceries": 15000, "Transport": 10000, "Eating_Out": 12000,
            "Entertainment": 10000, "Utilities": 5000, "Healthcare": 3000,
            "Education": 5000, "Miscellaneous": 10000, "Dependents": 2
        })
    ]

    print("\n=== SANITY TESTS ===")
    for name, profile in test_cases:
        print(f"\n--- {name} ---")
        rec = assistant.recommend(profile)
        conf = assistant.confidence(profile)
        print(json.dumps(rec, indent=2))
        print(f"Confidence: {conf:.3f}")

# ============================================================
# 5. MAIN
# ============================================================
if __name__ == "__main__":
    train_pipeline()
    run_sanity_tests()